In [1]:

import torch
import triton
import triton.language as tl



In [2]:
!nvidia-smi

Wed Nov 11 12:17:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.99.02              Driver Version: 595.99.02      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   61C    P8              2W /   42W |      12MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
@triton.jit
def add_ten_kernel(
    A, B,
    M: tl.constexpr,
    N: tl.constexpr,
    stride_am: tl.constexpr,
    stride_an: tl.constexpr,
    stride_bm: tl.constexpr,
    stride_bn: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # TODO 1: Calculate global row indices
    rows = pid_m*BLOCK_M + tl.arange(0,BLOCK_M)

    # TODO 2: Calculate global column indices
    cols = pid_n*BLOCK_N + tl.arange(0,BLOCK_N)

    # TODO 3: Construct the 2D input pointers
    a_ptrs = A + (rows[:,None]*stride_am+cols[None,:]*stride_an)

    # TODO 4: Construct the 2D output pointers
    b_ptrs = B + (rows[:,None]*stride_bm+cols[None,:]*stride_bn)

    # TODO 5: Construct the 2D mask
    mask = ((rows[:,None]<M) & (cols[None,:]<N))

    # Load matrix tile
    values = tl.load(a_ptrs, mask=mask, other=0)

    # Add 10
    result = values + 10

    # TODO 6: Store the result
    tl.store(b_ptrs,result,mask=mask)
    

    


In [4]:

def check(A):
    M, N = A.shape
    B = torch.empty((M, N), device=A.device, dtype=A.dtype)

    grid = (
        triton.cdiv(M, 4),
        triton.cdiv(N, 4),
    )

    add_ten_kernel[grid](
        A, B,
        M, N,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        4, 4,
    )

    torch.testing.assert_close(B, A + 10)
    print("PASSED")
    print(B)


# Test 1: Contiguous matrix
A = torch.arange(
    35, device="cuda", dtype=torch.float32
).reshape(5, 7)

check(A)


PASSED
tensor([[10., 11., 12., 13., 14., 15., 16.],
        [17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30.],
        [31., 32., 33., 34., 35., 36., 37.],
        [38., 39., 40., 41., 42., 43., 44.]], device='cuda:0')


In [14]:

@triton.jit
def transpose_kernel(
    A, B,
    M: tl.constexpr,
    N: tl.constexpr,
    stride_am: tl.constexpr,
    stride_an: tl.constexpr,
    stride_bm: tl.constexpr,
    stride_bn: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    # Task 1: Input tile pointers
    a_ptrs = A + (rows[:,None]*stride_am + cols[None,:]*stride_an)

    # Task 2: Transposed output pointers
    b_ptrs = B + (cols[None,:]*stride_bm + rows[:,None]*stride_bn)

    # Task 3: Boundary mask
    mask = ((rows[:,None]<M) & (cols[None,:]<N))
   

    # Task 4: Load
    values = tl.load(a_ptrs,mask=mask,other=0.0)

    # Task 5: Store transposed data
    tl.store(b_ptrs,values,mask=mask)


In [15]:
A = torch.arange(
    12, device="cuda", dtype=torch.float32
).reshape(3, 4)

B = torch.empty(
    (4, 3), device="cuda", dtype=A.dtype
)

grid = (triton.cdiv(3, 4), triton.cdiv(4, 4))

transpose_kernel[grid](
    A, B,
    3, 4,
    *A.stride(),
    *B.stride(),
    4, 4,
)

torch.testing.assert_close(B, A.T)
print("PASSED")

PASSED


### Flattened Indexing

In [25]:

import torch
import triton
import triton.language as tl


@triton.jit
def transform_kernel(
    A,
    B,
    M: tl.constexpr,
    N: tl.constexpr,
    stride_m: tl.constexpr,
    stride_n: tl.constexpr,
    BLOCK: tl.constexpr,
):
    # TODO 1:
    # Generate flattened indices for this program.
    p = tl.program_id(0)*BLOCK + tl.arange(0,BLOCK)

    # TODO 2:
    # Convert flattened indices into 2D coordinates.
    row = p//N
    col = p%N

    # TODO 3:
    # Calculate physical offsets for A using strides.
    input_offset = row*stride_m + col*stride_n

    # TODO 4:
    # Build a mask for valid logical elements.
    mask = p<M*N # ((row<M)&(col<N))

    # TODO 5:
    # Load values from A.
    x = tl.load(A+input_offset,mask=mask,other=0.0)

    # TODO 6:
    # Compute B[i,j] = 2 * A[i,j] + 5
    y = 2*x+5

    # TODO 7:
    # Store into contiguous B using flattened indices.
    tl.store(B+p,y,mask=mask)


def transform(A):
    M, N = A.shape
    B = torch.empty(
        (M, N),
        device=A.device,
        dtype=A.dtype
    )

    BLOCK = 128

    # TODO 8:
    # Calculate the 1D launch grid.
    # M, N = A.shape
    grid = (triton.cdiv(M*N,BLOCK),)

    transform_kernel[grid](
        A, B,
        M, N,
        A.stride(0),
        A.stride(1),
        BLOCK,
    )

    return B


In [26]:

def check(A, name):
    B = transform(A)
    expected = 2 * A + 5

    torch.testing.assert_close(B, expected)

    print(f"PASS: {name}")
    print(f"Shape: {A.shape}")
    print(f"Strides: {A.stride()}")
    print(f"Output:\n{B}\n")


# Test 1: Contiguous input
A1 = torch.arange(
    12, device="cuda", dtype=torch.float32
).reshape(3, 4)

check(A1, "Contiguous matrix")


# Test 2: Noncontiguous input
base = torch.arange(
    24, device="cuda", dtype=torch.float32
).reshape(3, 8)

A2 = base[:, ::2]

check(A2, "Noncontiguous matrix")


# Test 3: Transposed input
A3 = torch.arange(
    20, device="cuda", dtype=torch.float32
).reshape(4, 5).T

check(A3, "Transposed matrix")


# Test 4: Irregular dimensions
A4 = torch.randn(
    (17, 23), device="cuda"
)

check(A4, "Irregular dimensions")


# Test 5: Broadcasted input
A5 = torch.arange(
    5, device="cuda", dtype=torch.float32
).unsqueeze(0).expand(4, 5)

check(A5, "Broadcasted matrix")


PASS: Contiguous matrix
Shape: torch.Size([3, 4])
Strides: (4, 1)
Output:
tensor([[ 5.,  7.,  9., 11.],
        [13., 15., 17., 19.],
        [21., 23., 25., 27.]], device='cuda:0')

PASS: Noncontiguous matrix
Shape: torch.Size([3, 4])
Strides: (8, 2)
Output:
tensor([[ 5.,  9., 13., 17.],
        [21., 25., 29., 33.],
        [37., 41., 45., 49.]], device='cuda:0')

PASS: Transposed matrix
Shape: torch.Size([5, 4])
Strides: (1, 5)
Output:
tensor([[ 5., 15., 25., 35.],
        [ 7., 17., 27., 37.],
        [ 9., 19., 29., 39.],
        [11., 21., 31., 41.],
        [13., 23., 33., 43.]], device='cuda:0')

PASS: Irregular dimensions
Shape: torch.Size([17, 23])
Strides: (23, 1)
Output:
tensor([[ 1.9870,  5.3598,  6.0747,  5.3093,  5.4790,  8.8436,  2.7765,  5.7783,
          9.0609,  4.9079,  3.6257,  5.7700,  5.8055,  5.0092,  7.4678,  1.5341,
          5.6047,  6.0643,  5.4102,  3.4205,  5.4901,  2.5157,  4.0527],
        [ 2.0872,  3.5029,  5.6755, 10.4561,  5.7829, 10.2376,  2.3675,  